<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/5_deepeval_evaluation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## evaluation to csv and excel

In [ ]:
import json
import os
import pandas as pd


def eval_to_tab(
    json_path: str, output_prefix: str
    ):
    # Extract file stem to use in the ID
    file_stem = os.path.splitext(os.path.basename(json_path))[0]

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    # Iterate through test cases
    for test_case in data.get("conversationalTestCases", []):
        order = test_case.get("order", "")

        # Extract metric details (each test case has 4 metrics)
        for metric in test_case.get("metricsData", []):
            metric_name = metric.get("name")
            rows.append(
                {
                    "id": f"{file_stem}_{order}_{metric_name}",
                    "metric name": metric_name,
                    "metric success": metric.get("success"),
                    "metric score": metric.get("score"),
                    "metric reason": metric.get("reason"),
                }
            )

    df = pd.DataFrame(rows)

    # Save to CSV
    csv_file = f"{output_prefix}.csv"
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")

    # Save to Excel
    excel_file = f"{output_prefix}.xlsx"
    df.to_excel(excel_file, index=False)

    print(f"Export completed successfully:")
    print(f"- {csv_file}")
    print(f"- {excel_file}")
    return df


eval_to_tab("test_conversational_skills.json","test_conversatioanl_skills")

## summary sheet for each evaluation

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


def add_summary_sheet(
    excel_path: str,
    source_sheet: str = "Sheet1",
    summary_sheet: str = "Summary_Stats",
):
    # Read the raw evaluation data
    df = pd.read_excel(excel_path, sheet_name=source_sheet)

    summary_rows = []

    # Group by metric name and calculate statistics
    for metric_name, group in df.groupby("metric name", sort=False):
        scores = group["metric score"].dropna().values
        n = len(scores)

        if n < 2:
            continue

        mean = np.mean(scores)
        std_dev = np.std(scores, ddof=1)  # Sample standard deviation (ddof=1)
        min_val = np.min(scores)
        max_val = np.max(scores)

        # 95% Confidence Interval & Margin of Error (t-distribution)
        t_crit = stats.t.ppf(1 - 0.05 / 2, df=n - 1)
        moe = t_crit * (std_dev / np.sqrt(n))
        ci_lower = mean - moe
        ci_upper = mean + moe

        summary_rows.append(
            {
                "Metric Name": metric_name,
                "Count (n)": n,
                "Mean": round(mean, 4),
                "Min": round(min_val, 4),
                "Max": round(max_val, 4),
                "Std Deviation": round(std_dev, 4),
                "MoE (95%)": round(moe, 4),
                "95% CI Lower": round(ci_lower, 4),
                "95% CI Upper": round(ci_upper, 4),
            }
        )

    summary_df = pd.DataFrame(summary_rows)

    # Append / overwrite the new sheet into the existing Excel file
    with pd.ExcelWriter(
        excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
    ) as writer:
        summary_df.to_excel(writer, sheet_name=summary_sheet, index=False)

    print(f"Summary sheet '{summary_sheet}' added successfully to {excel_path}")
    return summary_df


# Example usage:
add_summary_sheet("test_conversatioanl_skills.xlsx")

## intra-criterion pearson correlation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns


def calculate_intra_criterion_correlation(
    excel_files: list,
    output_prefix: str,
    sheet_name: str = "Sheet1",
):
    """Loads long-format evaluation files, pivots to wide format,

    computes Pearsoncorrelation matrices, p-values, and saves heatmaps.
    """
    # Load and combine all domain datasets (Jobs, Occupations, Skills)
    dfs = [pd.read_excel(f, sheet_name=sheet_name) for f in excel_files]
    df = pd.concat(dfs, ignore_index=True)

    # Extract base test_case_id (strips the metric suffix if present)
    if "test_case_id" not in df.columns:
        df["test_case_id"] = df["id"].apply(
            lambda x: "_".join(str(x).split("_")[:4])
        )

    # Pivot: One row per test case, one column per metric
    pivot_df = df.pivot(
        index="test_case_id",
        columns="metric name",
        values="metric score",
    )

    # Compute Pearson (linear) and Spearman (rank) correlation
    pearson_corr = pivot_df.corr(method="pearson")
    spearman_corr = pivot_df.corr(method="spearman")

    # Compute P-Value Significance Matrix (for Pearson)
    p_matrix = pd.DataFrame(
        np.zeros_like(pearson_corr),
        index=pearson_corr.index,
        columns=pearson_corr.columns,
    )
    for c1 in pivot_df.columns:
        for c2 in pivot_df.columns:
            if c1 == c2:
                p_matrix.loc[c1, c2] = 0.0
            else:
                subset = pivot_df[[c1, c2]].dropna()
                _, p_val = stats.pearsonr(subset[c1], subset[c2])
                p_matrix.loc[c1, c2] = p_val

    # Save Correlation Matrix to Excel
    excel_out = f"Intra_Criterion_Correlation_{output_prefix}.xlsx"
    with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
        pearson_corr.to_excel(writer, sheet_name="Pearson_Corr")
        spearman_corr.to_excel(writer, sheet_name="Spearman_Corr")
        p_matrix.to_excel(writer, sheet_name="P_Values")
        pivot_df.to_excel(writer, sheet_name="Pivoted_Data")

    # Generate Publication-Ready Heatmap for Thesis
    plt.figure(figsize=(7, 6))
    mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)
    sns.heatmap(
        pearson_corr,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    plt.title(
        f"Intra-Criterion Correlation Matrix ({output_prefix.replace('_', ' ').title()})",
        fontsize=12,
        fontweight="bold",
        pad=12,
    )
    plt.tight_layout()
    plt.savefig(
        f"correlation_heatmap_{output_prefix}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

    print(
        f"Successfully generated correlation analysis for {output_prefix}:"
    )
    print(f"- Excel: {excel_out}")
    print(f"- Heatmap: correlation_heatmap_{output_prefix}.png\n")

    return pearson_corr, p_matrix




# Single Turn

single_turn_files = [
    "test_single_jobs.xlsx",
    "test_single_occupations.xlsx",
    "test_single_skills.xlsx",
]
single_corr, single_p = calculate_intra_criterion_correlation(
    excel_files=single_turn_files, output_prefix="single_turn"
)


# Multi-Turn

multi_turn_files = [
    "test_conversatioanl_jobs.xlsx",
    "test_conversatioanl_occupations.xlsx",
    "test_conversatioanl_skills.xlsx",
]
multi_corr, multi_p = calculate_intra_criterion_correlation(
    excel_files=multi_turn_files, output_prefix="multi_turn"
)